In [1]:
%%capture
!pip install transformers

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load pre-trained model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Successfully loaded model: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Successfully loaded model: gpt2


In [5]:
def generate_response(prompt, model, tokenizer, chat_history_ids=None):
    # Encode the new user input
    new_input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')

    # Append new input to chat history if it exists, otherwise use new input as history
    if chat_history_ids is None:
        input_ids = new_input_ids
    else:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)

    chat_history_ids = model.generate(
        input_ids,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1
    )

    response = tokenizer.decode(chat_history_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response, chat_history_ids

print("Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to end the conversation)")

chat_history_ids = None # Initialize chat history
conversation_count = 0

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    # Generate response and update chat history
    response, chat_history_ids = generate_response(user_input, model, tokenizer, chat_history_ids)

    print(f"Chatbot: {response}")

    conversation_count += 1
    if conversation_count > 10:
        print("\nChatbot: I'm resetting our conversation history to keep things fresh. How else can I help?")
        chat_history_ids = None
        conversation_count = 0


Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to end the conversation)
You: hello
Chatbot: How did a black man get into the White House?

Donald Trump's campaign manager said Tuesday he was confident that the GOP presidential nominee would meet the president on the phone to talk about Russia's alleged interference in the election in 2016.

"His goal is to get a chance to get a meeting with the president. He's not going to do it for $50,000 per call," Ryan Howard told MSNBC's "Morning Joe."

Mr. Trump's campaign declined to answer
You: lol
Chatbot: This book is a little strange considering the recent election, which had a major focus on the Obama presidency and his administration. I'd like to point out that Obama's campaign slogan, "To the moon, to man", may well have been "We are with you!" But if Obama were to continue pursuing his political agenda and his policies for the rest of his life, there were many questions that are being raised over

In [ ]:
prompt = "Hello! I am your AI assistant. How can I help you today?"

# Encode the prompt
input_ids = tokenizer.encode(prompt, return_tensors='pt')


output = model.generate(input_ids, max_new_tokens=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

# Decode the generated output
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("Chatbot Initial Response:")
print(generated_text)
